# Extracting Activations and Covariances from ResNet18

This tutorial walks through how to use `cov_extractor` and `cov_extractor_batch` to pull activations and covariance matrices from named layers of a pretrained ResNet18.

We will cover three use cases:

1. **Activations** — raw layer outputs for a batch of images  
2. **Covariances** — covariance matrices computed across images  
3. **Gradient-enabled mode** — keeping the outputs attached to the computation graph for backpropagation

> **Prerequisites:** familiarity with PyTorch modules, tensors, and named children.


##### Imports

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models

from BayesCompare import cov_extractor, cov_extractor_batch, get_layer_names

##### Load the PretrainedResNet18

We load a standard ResNet18. We do **not** need pretrained weights for this tutorial — the extraction logic is weight-agnostic — but you can swap in `weights=ResNet18_Weights.DEFAULT` if you want realistic activations.


In [2]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### 1. Inspecting available layer names and defining the wanted layers

To get the usable layer names for `cov_extractor` and `cov_extractor_batch`, we will use the function `get_layer_names` from `BayesCompare`. All `dnn_extract` functions of `BayesCompare` use `TorchLens` under the hood. Therefore, layer naming conventions are based on `TorchLens` naming. Please refer to [TorchLens Repo](https://github.com/johnmarktaylor91/torchlens) for more details.


In [3]:
layer_names = get_layer_names(model)

for name in layer_names:
    print(f"{name:40s}")

input_1                                 
conv2d_1_1                              
buffer_1                                
buffer_2                                
batchnorm_1_2                           
relu_1_3                                
maxpool2d_1_4                           
conv2d_2_5                              
buffer_3                                
buffer_4                                
batchnorm_2_6                           
relu_2_7                                
conv2d_3_8                              
buffer_5                                
buffer_6                                
batchnorm_3_9                           
iadd_1_10                               
relu_3_11                               
conv2d_4_12                             
buffer_7                                
buffer_8                                
batchnorm_4_13                          
relu_4_14                               
conv2d_5_15                             
buffer_9        

We will target three layers at different depths:

| Layer name | Module type |
|---|---|
| `conv2d_1_1` | Conv2d |
| `relu_7_27` | ReLU |
| `linear_1_69` | Linear |


In [4]:
layer_list = ["conv2d_1_1", "relu_7_27", "linear_1_69"]

### 2. Get your input images

We generate a batch of random RGB images that match ResNet's expected input shape `(N, 3, 224, 224)`. We are using seed for reproducibility. Feel free to use your favourite image set instead.

In [5]:
N_IMAGES = 32
IMG_SHAPE = (N_IMAGES, 3, 224, 224)

rng = np.random.default_rng(42)
input_images = rng.standard_normal(IMG_SHAPE).astype(np.float32)

### Alternative: Load your favourite image set
"""
import PIL
import os

image_set_path = "" # put here the path of your image set
file_names = os.listdir(image_set_path)
ims = [PIL.Image.open(os.path.join(image_set_path, f_name)) for f_name in file_names[:N_IMAGES]]

transforms = models.ResNet18_Weights.IMAGENET1K_V1.transforms()
transformed_ims = [transforms(im.convert("RGB")) for im in ims]
input_images = torch.stack(transformed_ims)
"""

print(f"Input shape : {input_images.shape}")
print(f"dtype       : {input_images.dtype}")
print(f"value range : [{input_images.min():.3f}, {input_images.max():.3f}]")

Input shape : (32, 3, 224, 224)
dtype       : float32
value range : [-5.221, 5.312]


### 3. Extracting activations with `cov_extractor`

Setting `compute_covs=False` returns the raw activations for each layer.

In [6]:
activations = cov_extractor(
    model,
    input_images,
    layer_list,
    compute_covs=False,
)

for name, act in activations.items():
    print(f"{name:30s}  shape: {tuple(act.shape)}")

conv2d_1_1                      shape: (32, 64, 112, 112)
relu_7_27                       shape: (32, 128, 28, 28)
linear_1_69                     shape: (32, 1000)


Each value is a `torch.Tensor` of shape `(N, C, H, W)` — one row per image, preserving the spatial dimensions, except for the last layer `linear_1_69` which has dimesions `(1, 1000)` corresponding to 1000 classes this ResNet18 was trained on.

> **Note:** The spatial dimensions `H` and `W` depend on where in the network the layer sits. Deeper layers have smaller feature maps due to strided convolutions and pooling.

It is also possible to flatten all the activations per layer so that it is of shape `(N, C*H*W)` using the `flatten_acts` argument. We reshape activations to this shape before computing covariance, therefore, it may be handy to have them already in this shape.

In [7]:
activations = cov_extractor(
    model,
    input_images,
    layer_list,
    compute_covs=False,
    flatten_acts=True
)

for name, act in activations.items():
    print(f"{name:30s}  shape: {tuple(act.shape)}")

conv2d_1_1                      shape: (32, 802816)
relu_7_27                       shape: (32, 100352)
linear_1_69                     shape: (32, 1000)


### 4. Extracting covariances with `cov_extractor`

We can get covariances by simply providing `cov_extractor` with the model, input images and the list of layers we are interested in. Covariance is calculated from the activations by first flattening the activations per layer so that they would be of the shape `(N, C*H*W)`. Then we compute the covariance with `activation @ activation.T`. Therefore the resulting covariance matrix per layer is of shape `(N, N)`.

In [8]:
covs = cov_extractor(
    model,
    input_images,
    layer_list,
    # compute_covs is True by default so we don't need to indicate
)


for name, cov in covs.items():
    print(f"{name:30s}  shape: {tuple(cov.shape)}")

conv2d_1_1                      shape: (32, 32)
relu_7_27                       shape: (32, 32)
linear_1_69                     shape: (32, 32)


#### Quick sanity checks

Let's verify that the covariance matrices are symmetric and positive semi-definite.


In [9]:
for name, cov in covs.items():
    cov_np = cov.numpy()

    # Symmetry
    sym_err = np.max(np.abs(cov_np - cov_np.T))

    # Smallest eigenvalue (should be >= 0)
    min_eig = np.linalg.eigvalsh(cov_np).min()

    print(f"{name:30s}  max |C - C^T|: {sym_err:.2e}  min eigenvalue: {min_eig:.4f}")

conv2d_1_1                      max |C - C^T|: 5.86e-03  min eigenvalue: 1869499.2500
relu_7_27                       max |C - C^T|: 1.46e-03  min eigenvalue: 3107.2551
linear_1_69                     max |C - C^T|: 0.00e+00  min eigenvalue: 3.7777


### 5. Gradient-enabled mode

In some cases we may need the gradients to be computed from the activations or covariances we extracted. For such cases, we can use the input argument `gradient=True` in `cov_extractor`. As a result, `cov_extractor` runs the model with gradient tracking enabled and forces the model into training mode. The returned tensors remain attached to the computation graph, so you can backpropagate through them.

This is useful when the covariances or activations are part of a loss function, for example if we want to define a loss with one of the BayesCompare distances or other methods such as representational similarity analysis (RSA).

In [10]:
# Convert inputs to a tensor so gradients can flow through them too
images_tensor = torch.from_numpy(input_images)

covs_grad = cov_extractor(
    model,
    images_tensor,
    layer_list,
    gradient=True,   # keeps outputs in the graph
)

for name, cov in covs_grad.items():
    print(f"{name:30s}  requires_grad: {cov.requires_grad}  grad_fn: {cov.grad_fn}")


conv2d_1_1                      requires_grad: True  grad_fn: <MmBackward0 object at 0x7e320d9fb910>
relu_7_27                       requires_grad: True  grad_fn: <MmBackward0 object at 0x7e3204b57760>
linear_1_69                     requires_grad: True  grad_fn: <MmBackward0 object at 0x7e3204b57760>


#### Example: backpropagating through a covariance-based loss

As a concrete illustration, we define a toy loss — the squared Frobenius norm of the covariance of the first targeted layer — and backpropagate through it.


In [11]:
# Use only the first layer's covariance for this example
cov_target = covs_grad[layer_list[0]]

# Toy loss: squared Frobenius norm of the covariance matrix
loss = (cov_target ** 2).sum()
print(f"Loss value : {loss.item():.4f}")

loss.backward()

# Check that model parameters received gradients
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"  {name:40s}  grad norm: {param.grad.norm().item():.4f}")
        break  # just show the first one to keep output short


Loss value : 124091528904704.0000
  conv1.weight                              grad norm: 39466861330432.0000


Beware that by default, `cov_extractor` sets `gradient=False`. Therefore, in the default behaviour, `cov_extractor` will not be keeping the activations/covariances attached to the model graph.

In [12]:
covs_wo_grad = cov_extractor(
    model,
    images_tensor,
    layer_list,
)

for name, cov in covs_wo_grad.items():
    print(f"{name:30s}  shape: {tuple(cov.shape)}  requires_grad: {cov.requires_grad}")

conv2d_1_1                      shape: (32, 32)  requires_grad: False
relu_7_27                       shape: (32, 32)  requires_grad: False
linear_1_69                     shape: (32, 32)  requires_grad: False


However, it is optional to get model covariances in the eval mode `model.eval()` or within the `torch.inference_mode()`. When at least one of `eval_mode` or `inference_mode` is `True`, the activations/covariances will be detached from the graph. You may also set a random seed for reproducible results with stochastic models.

In [13]:
covs_in_eval_and_inference = cov_extractor(
    model,
    input_images,
    layer_list,
    compute_covs=True,     # compute covariances (default)
    gradient=False,        # no gradient tracking (default)
    eval_mode=True,        # model.eval() (default)
    inference_mode=True,   # torch.inference_mode() (default)
    random_seed=88,        # seed for reproducible results
)

print("Covariances obtained when model is in eval and within torch.inference_mode()\n")
for name, cov in covs_in_eval_and_inference.items():
    print(f"{name:30s}  shape: {tuple(cov.shape)}  requires_grad: {cov.requires_grad}")
    
print("\n")

covs_without_eval_and_inference = cov_extractor(
    model,
    input_images,
    layer_list,
    compute_covs=True,     # compute covariances (default)
    gradient=False,        # no gradient tracking (default)
    eval_mode=False,       # model.eval() (default)
    inference_mode=False,  # torch.inference_mode() (default)
    random_seed=88,        # seed for reproducible results
)
print("Covariances obtained when model is NOT in eval and WITHOUT torch.inference_mode()\n")
for name, cov in covs_without_eval_and_inference.items():
    print(f"{name:30s}  shape: {tuple(cov.shape)}  requires_grad: {cov.requires_grad}")

Covariances obtained when model is in eval and within torch.inference_mode()

conv2d_1_1                      shape: (32, 32)  requires_grad: False
relu_7_27                       shape: (32, 32)  requires_grad: False
linear_1_69                     shape: (32, 32)  requires_grad: False


Covariances obtained when model is NOT in eval and WITHOUT torch.inference_mode()

conv2d_1_1                      shape: (32, 32)  requires_grad: True
relu_7_27                       shape: (32, 32)  requires_grad: True
linear_1_69                     shape: (32, 32)  requires_grad: True


### 6. Large-scale extraction with `cov_extractor_batch`

For large image sets that don't fit in memory, or a large set of wanted layers, `cov_extractor_batch` processes inputs in chunks and saves results to disk with the given name using the `out_filename` argument as:

- **HDF5** (`.hdf5`) for activations  
- **Pickle** (`.pkl`) for covariances  

Results can be saved either as a single combined file or as one file per layer (`layer_by_layer=True`). 

##### Single, combined file:

- The **HDF5** (`.hdf5`) file is saved as `activations_{out_filename}.hdf5` and has datasets `activations_ + layer_name` .
- The **Pickle** (`.pkl`) file is saved as `covs_{out_filename}.pkl` has a dictionary where keys are the layer names and each value is the corrsponding covariance matrix of that layer.

##### Layer-by-layer saving:

- Each **HDF5** file is named as `activations_{out_filename}_{layer_name}.hdf5`. Each **HDF5** file has only one dataset named `activations`.
- Each **Pickle** file is named as `covs_{out_filename}_{layer_name}.pkl`. Each **Pickle** file has only a tensor/numpy array corresponding to that layer.

> **Note:** When one wants to save only covariance results, it is possible to delete all the created **HDF5** files by using `delete_act_files=True`. This argument is only considered if `compute_covs=True`.

#### 6a. Saving covariances of all layers to a single Pickle file

In [21]:
import os, pickle, h5py

OUT_DIR = "./tutorial_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# --- Covariances: single combined pickle ---
cov_extractor_batch(
    model,
    input_images,
    layer_list,
    out_filename="resnet18_demo",
    out_dir=OUT_DIR,
    batch_size=8,
    layer_by_layer=False,   # all layers in one file (default)
    compute_covs=True,
    delete_act_files=True,  # remove intermediate HDF5 after computing covs
    eval_mode=True,
    random_seed=42,
)

# Load and inspect
with open(os.path.join(OUT_DIR, "covs_resnet18_demo.pkl"), "rb") as f:
    saved_covs = pickle.load(f)

print("Keys in saved pickle:", list(saved_covs.keys()))
for name, cov in saved_covs.items():
    print(f"  {name:30s}  shape: {cov.shape}")


Batches - Activation Extraction - All Layers: 100%|██████████| 4/4 [00:01<00:00,  2.35it/s]

All activations are saved to './tutorial_outputs/activations_resnet18_demo.hdf5'

Saved covariance for layer linear_1_69 at ./tutorial_outputs/covs_resnet18_demo.pkl.

Deleted the activation file at ./tutorial_outputs/activations_resnet18_demo.hdf5.
Keys in saved pickle: ['conv2d_1_1', 'relu_7_27', 'linear_1_69']
  conv2d_1_1                      shape: (32, 32)
  relu_7_27                       shape: (32, 32)
  linear_1_69                     shape: (32, 32)


#### 6b. Saving activations of all layers to a single HDF5 file

In [22]:
cov_extractor_batch(
    model,
    input_images,
    layer_list,
    out_filename="resnet18_demo_acts",
    out_dir=OUT_DIR,
    batch_size=8,
    compute_covs=False,   # save raw activations instead
    eval_mode=True,
    random_seed=42,
    flatten_acts=True
)

with h5py.File(os.path.join(OUT_DIR, "activations_resnet18_demo_acts.hdf5"), "r") as f:
    for name in f.keys():
        print(f"  {name:30s}  shape: {f[name].shape}")


Batches - Activation Extraction - All Layers: 100%|██████████| 4/4 [00:01<00:00,  2.48it/s]

All activations are saved to './tutorial_outputs/activations_resnet18_demo_acts.hdf5'
  activations_conv2d_1_1          shape: (32, 802816)
  activations_linear_1_69         shape: (32, 1000)
  activations_relu_7_27           shape: (32, 100352)


#### 6c. Saving one covariances file per layer (`layer_by_layer=True`)

In [23]:
cov_extractor_batch(
    model,
    input_images,
    layer_list,
    out_filename="resnet18_demo",
    out_dir=OUT_DIR,
    batch_size=8,
    layer_by_layer=True,    # separate file per layer
    compute_covs=True,
    delete_act_files=True,
    eval_mode=True,
    random_seed=42,
)

# Each layer gets its own pickle: covs_resnet18_demo_{layer_name}.pkl
for layer in layer_list:
    path = os.path.join(OUT_DIR, f"covs_resnet18_demo_{layer}.pkl")
    with open(path, "rb") as f:
        cov = pickle.load(f)
    print(f"  {layer:30s}  shape: {cov.shape}")


Batches - Activation Extraction - Layer conv2d_1_1: 100%|██████████| 4/4 [00:03<00:00,  1.26it/s]


Activations for layer conv2d_1_1 are saved at ./tutorial_outputs/activations_resnet18_demo_conv2d_1_1.hdf5.


Batches - Activation Extraction - Layer relu_7_27: 100%|██████████| 4/4 [00:02<00:00,  1.47it/s]


Activations for layer relu_7_27 are saved at ./tutorial_outputs/activations_resnet18_demo_relu_7_27.hdf5.


Batches - Activation Extraction - Layer linear_1_69: 100%|██████████| 4/4 [00:03<00:00,  1.33it/s]

Activations for layer linear_1_69 are saved at ./tutorial_outputs/activations_resnet18_demo_linear_1_69.hdf5.
All activations are saved to HDF5 files at the directory ./tutorial_outputs

Saved covariance for layer conv2d_1_1 at ./tutorial_outputs/covs_resnet18_demo_conv2d_1_1.pkl.

Deleted the activation file for layer conv2d_1_1 at ./tutorial_outputs/activations_resnet18_demo_conv2d_1_1.hdf5.

Saved covariance for layer relu_7_27 at ./tutorial_outputs/covs_resnet18_demo_relu_7_27.pkl.

Deleted the activation file for layer relu_7_27 at ./tutorial_outputs/activations_resnet18_demo_relu_7_27.hdf5.

Saved covariance for layer linear_1_69 at ./tutorial_outputs/covs_resnet18_demo_linear_1_69.pkl.

Deleted the activation file for layer linear_1_69 at ./tutorial_outputs/activations_resnet18_demo_linear_1_69.hdf5.
  conv2d_1_1                      shape: (32, 32)
  relu_7_27                       shape: (32, 32)
  linear_1_69                     shape: (32, 32)


#### 6d. Saving one activation file per layer (`layer_by_layer=True`)

In [24]:
cov_extractor_batch(
    model,
    input_images,
    layer_list,
    out_filename="resnet18_demo_acts",
    out_dir=OUT_DIR,
    batch_size=8,
    compute_covs=False,   # save raw activations instead
    layer_by_layer=True,
    eval_mode=True,
    random_seed=42,
    flatten_acts=False
)

for layer in layer_list:
    path = os.path.join(OUT_DIR, f"activations_resnet18_demo_acts_{layer}.hdf5")
    with h5py.File(path, "r") as f:
        for name in f.keys():
            print(f"File {path} has the activations:\n")
            print(f"  {name:30s}  shape: {f[name].shape}")

Batches - Activation Extraction - Layer conv2d_1_1: 100%|██████████| 4/4 [00:03<00:00,  1.20it/s]


Activations for layer conv2d_1_1 are saved at ./tutorial_outputs/activations_resnet18_demo_acts_conv2d_1_1.hdf5.


Batches - Activation Extraction - Layer relu_7_27: 100%|██████████| 4/4 [00:03<00:00,  1.28it/s]


Activations for layer relu_7_27 are saved at ./tutorial_outputs/activations_resnet18_demo_acts_relu_7_27.hdf5.


Batches - Activation Extraction - Layer linear_1_69: 100%|██████████| 4/4 [00:03<00:00,  1.32it/s]

Activations for layer linear_1_69 are saved at ./tutorial_outputs/activations_resnet18_demo_acts_linear_1_69.hdf5.
All activations are saved to HDF5 files at the directory ./tutorial_outputs
File ./tutorial_outputs/activations_resnet18_demo_acts_conv2d_1_1.hdf5 has the activations:

  activations                     shape: (32, 64, 112, 112)
File ./tutorial_outputs/activations_resnet18_demo_acts_relu_7_27.hdf5 has the activations:

  activations                     shape: (32, 128, 28, 28)
File ./tutorial_outputs/activations_resnet18_demo_acts_linear_1_69.hdf5 has the activations:

  activations                     shape: (32, 1000)


#### Consistency check: `cov_extractor` vs `cov_extractor_batch`

The two functions should produce numerically identical covariances for the same model and inputs.


In [25]:
# Reload the single-file batch result
with open(os.path.join(OUT_DIR, "covs_resnet18_demo.pkl"), "rb") as f:
    batch_covs = pickle.load(f)

print(f"{'Layer':30s}  {'Realtive diff':>15s}  {'Match?':>8s}")
print("-" * 60)
for name in layer_list:
    diff = np.max(np.abs(covs[name].numpy() - batch_covs[name]))
    max_value_in_covs = np.max([np.max(covs[name].numpy()), np.max(batch_covs[name])])
    relative_diff = diff/max_value_in_covs
    match = "✓" if relative_diff < 1e-4 else "✗"
    print(f"{name:30s}  {relative_diff:15.2e}  {match:>8s}")


Layer                             Realtive diff    Match?
------------------------------------------------------------
conv2d_1_1                             1.70e-06         ✓
relu_7_27                              7.59e-07         ✓
linear_1_69                            4.38e-07         ✓


## Summary

| Goal | Function | Key flags |
|---|---|---|
| Activations, in-memory | `cov_extractor` | `compute_covs=False` |
| Covariances, in-memory | `cov_extractor` | `compute_covs=True` |
| Backprop through outputs | `cov_extractor` | `gradient=True` |
| Large datasets, save to disk | `cov_extractor_batch` | `batch_size`, `out_dir`, `out_filename` |
| Separate file per layer | `cov_extractor_batch` | `layer_by_layer=True` |
| Don't keep intermediate activation files | `cov_extractor_batch` | `delete_act_files=False` |

A few things to keep in mind:

- `gradient=True` forces `eval_mode=False` and `inference_mode=False` internally, regardless of what you pass.
- Layer names must match the `TorchLens` conventions. Use the `get_layer_names` function when targeting new architectures.
- `cov_extractor_batch` and `cov_extractor` produce numerically identical results; the batch version simply trades memory for disk I/O.
